# 04 — Spark SQL dan Data Analytics (7 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Theory 2 JP; practice 5 JP.** Compare equivalent DataFrame and SQL queries, aggregation, statistics and business interpretations. Dataset is synthetic.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
payments=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/payments.csv")
taxpayers=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/taxpayers.csv")
payments.createOrReplaceTempView("payments")
taxpayers.createOrReplaceTempView("taxpayers")


In [ ]:
by_region=spark.sql("""
SELECT t.region, count(distinct p.taxpayer_id) AS paying_taxpayers,
       sum(p.amount) AS total_payment, avg(p.amount) AS avg_payment
FROM payments p JOIN taxpayers t ON p.taxpayer_id=t.taxpayer_id
WHERE p.status='PAID'
GROUP BY t.region
ORDER BY total_payment DESC
""")
by_region.show(truncate=False)


In [ ]:
equivalent=(payments.filter("status = 'PAID'").join(taxpayers,"taxpayer_id")
  .groupBy("region")
  .agg(F.countDistinct("taxpayer_id").alias("paying_taxpayers"),
       F.sum("amount").alias("total_payment"), F.avg("amount").alias("avg_payment")))
equivalent.orderBy(F.desc("total_payment")).show()


In [ ]:
payments.filter("status='PAID'").select("amount").summary("count","mean","stddev","min","25%","50%","75%","max").show()
by_region.explain(mode="formatted")


### Lab tasks
1. Compute total PAID amount per month and channel.
2. Rank 10 taxpayers by valid payment amount using a SQL window function.
3. Compare SQL and DataFrame physical plans for the same aggregation.
4. Interpret why comparing sums across regions does **not** indicate compliance or risk.
5. Explain a limitation of the synthetic dataset.

**Deliverable:** analytical SQL notebook + 3 business findings with caveats.
